# 第 2 课作业：把小数放进有限位宽

这份作业对应 **第 2 课：数字硬件怎样保存 0.22？**

本题要把“有限 bit”真正变成可以计算的规则：有符号范围、固定小数点、rounding 和 saturation。

## 本题目标与固定规则

你要实现：

- signed_limits(total_bits)：给出有符号整数编码范围；
- quantize(x, total_bits, frac_bits)：缩放、舍入、saturation，再解码成 represented value。

本课固定使用 Python 的 round()，overflow policy 固定为 **saturation**，不是 wraparound。

## 先不用代码：用 5-bit 定点数算一遍

设：

- total_bits = 5
- frac_bits = 3

先回答：

1. 有符号整数编码的最小值和最大值是多少？
2. scale = 2^frac_bits 是多少？
3. 最小刻度是多少？
4. x = 0.70 时，缩放后是多少？round 后的 integer code 是多少？represented value 是多少？
5. 如果 x = 3.0，round 后的 code 超出了范围，saturation 后应该停在哪里？对应 represented value 是多少？

先把“真实值 → integer code → represented value”这条链走通，再写函数。

## Part A：实现 signed_limits()

### 这个函数做什么？

`signed_limits()` 计算“给定总 bit 数时，一个有符号整数编码能表示到哪里”。

这里还没有小数点，也还没有 quantization；只处理 **signed integer code 的范围**。

### 输入

- `total_bits`：整个有符号整数编码一共占多少个 bit，其中已经包含 sign bit。

### 输出

函数返回 **两个整数**，顺序固定为：

`(min_i, max_i)`

其中：

1. `min_i`：这个 bit width 能表示的最小 signed integer code；
2. `max_i`：这个 bit width 能表示的最大 signed integer code。

例如后面的 `quantize()` 会使用这两个边界判断缩放后的 integer code 是否溢出。

In [ ]:
def signed_limits(total_bits: int) -> tuple[int, int]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: compute the signed integer range")
    # YOUR CODE ENDS HERE

## Part B：实现 quantize()

### 这个函数做什么？

`quantize()` 把一个普通实数 `x` 放进指定的 fixed-point 格式中。

本题的流程已经固定为：

**scale → round → saturation → decode**

其中 `scale` 和 signed range 已经在 starter 中算好；你的 TODO 负责后面三步。

### 输入

- `x`：想要表示的原始实数；
- `total_bits`：fixed-point 编码总 bit 数，包含 sign bit；
- `frac_bits`：其中有多少个 bit 用来表示小数部分。

### 输出

函数返回 **两个值**，顺序固定为：

`(integer_code, represented_value)`

其中：

1. `integer_code`：一个整数，表示经过 scaling、rounding 和 saturation 后，真正存进有限位宽编码中的 integer code；
2. `represented_value`：一个浮点数，表示这个 integer code 解码回现实数轴以后，硬件**实际能够表示的值**。

`represented_value` 不一定等于原始 `x`；两者的差就是 quantization 带来的结果之一。

In [ ]:
def quantize(
    x: float,
    total_bits: int = 8,
    frac_bits: int = 4,
) -> tuple[int, float]:
    scale = 1 << frac_bits
    min_i, max_i = signed_limits(total_bits)

    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: round, saturate, and decode")
    # YOUR CODE ENDS HERE

    return integer_code, represented_value

## 检查你的实现

先运行两个实现单元，再运行外部 grader。Notebook 不显示具体测试向量。

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exercises" / "grader").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson02 import check

check(signed_limits=signed_limits, quantize=quantize, language="zh")

## Human Check

结合你自己的实现做“故障定位”，不要只复述定义：

1. 如果 integer_code 是对的，但 represented_value 错了，你会先检查 quantize() 的哪个阶段？为什么？
2. 如果一个很大的正输入最后变成了负数，最可能说明你违反了本课哪条 overflow contract？
3. 如果修改 frac_bits 后量化步长完全没有变化，你会检查哪个中间量？
4. 为了调试 quantize()，你最想临时观察哪三个中间值？说明每个值分别能排查哪类错误。